<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula11b_ajuste_de_hiperpar%C3%A2metros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import load_wine
import numpy as np

X, y = load_wine(return_X_y=True)
print(X.shape)
unique_labels, counts = np.unique(y, return_counts=True)
print("Label counts:")
for label, count in zip(unique_labels, counts):
    print(f"Label {label}: {count}")

(178, 13)
Label counts:
Label 0: 59
Label 1: 71
Label 2: 48


In [2]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

# Qual o valor ideal para K?

In [4]:
# Não pode usar o treino e o teste para achar os parâmetros,
# para não dar overfitting no teste.

from sklearn.neighbors import KNeighborsClassifier

### ERRADO !!!

model = KNeighborsClassifier()
def ajuste_errado(model, X_train, y_train, X_test, y_test):
  best_k = 0
  best_accuracy = 0
  for i in range(1, 15, 2):
      print(f"K = {i}")
      model.set_params(n_neighbors=i)
      model.fit(X_train, y_train)
      accuracy = model.score(X_test, y_test)
      print(f"Acurácia: {accuracy}")
      if accuracy > best_accuracy:
          best_accuracy = accuracy
          best_k = i
  print(f"Melhor K: {best_k}")
  print(f"Melhor Acurácia: {best_accuracy}")
  return model.set_params(n_neighbors=best_k)
ajuste_errado(model, X_train, y_train, X_test, y_test)

K = 1
Acurácia: 0.8055555555555556
K = 3
Acurácia: 0.8055555555555556
K = 5
Acurácia: 0.75
K = 7
Acurácia: 0.75
K = 9
Acurácia: 0.7777777777777778
K = 11
Acurácia: 0.75
K = 13
Acurácia: 0.7222222222222222
Melhor K: 1
Melhor Acurácia: 0.8055555555555556


KNeighborsClassifier(n_neighbors=1)

In [5]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.2)
ajuste_errado(model, X_train2, y_train2, X_test2, y_test2)

K = 1
Acurácia: 0.8055555555555556
K = 3
Acurácia: 0.8333333333333334
K = 5
Acurácia: 0.8055555555555556
K = 7
Acurácia: 0.8333333333333334
K = 9
Acurácia: 0.8055555555555556
K = 11
Acurácia: 0.7777777777777778
K = 13
Acurácia: 0.7777777777777778
Melhor K: 3
Melhor Acurácia: 0.8333333333333334


KNeighborsClassifier(n_neighbors=3)

In [28]:
# Correto (treino, validação e teste)

def ajuste_correto(model, X_train, y_train, X_test, y_test):
  best_k = 0
  best_accuracy = 0
  X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2)
  for i in range(1, 15, 2):
      print(f"K = {i}")
      model.set_params(n_neighbors=i)
      model.fit(X_tr, y_tr)
      accuracy = model.score(X_val, y_val)
      print(f"Acurácia: {accuracy}")
      if accuracy > best_accuracy:
          best_accuracy = accuracy
          best_k = i
  print(f"Melhor K: {best_k}")
  print(f"Melhor Acurácia: {best_accuracy}")
  model.set_params(n_neighbors=best_k)
  model.fit(X_train, y_train)
  return model.score(X_test, y_test)
print(f"\nAcurácia final: {ajuste_correto(model, X_train, y_train, X_test, y_test)}")

K = 1
Acurácia: 0.7241379310344828
K = 3
Acurácia: 0.7241379310344828
K = 5
Acurácia: 0.7241379310344828
K = 7
Acurácia: 0.7241379310344828
K = 9
Acurácia: 0.7586206896551724
K = 11
Acurácia: 0.7241379310344828
K = 13
Acurácia: 0.6896551724137931
Melhor K: 9
Melhor Acurácia: 0.7586206896551724

Acurácia final: 0.7777777777777778


In [65]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

def ajuste_com_validacao_cruzada(model, X_train, y_train, X_test, y_test):
  best_k = 0
  best_accuracy = 0
  for i in range(1, 15, 2):
      print(f"K = {i}")
      model.set_params(n_neighbors=i)
      score = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(n_splits=3, shuffle=True))
      print(f"Acurácia: {score.mean()}")
      if score.mean() > best_accuracy:
          best_accuracy = score.mean()
          best_k = i
  print(f"Melhor K: {best_k}")
  print(f"Melhor Acurácia: {best_accuracy}")
  model.set_params(n_neighbors=best_k)
  model.fit(X_train, y_train)
  return model.score(X_test, y_test)
print(f"\nAcurácia final: {ajuste_com_validacao_cruzada(model, X_train, y_train, X_test, y_test)}")

K = 1
Acurácia: 0.6900118203309692
K = 3
Acurácia: 0.6477541371158392
K = 5
Acurácia: 0.6759751773049646
K = 7
Acurácia: 0.6898640661938534
K = 9
Acurácia: 0.6971040189125296
K = 11
Acurácia: 0.6835106382978724
K = 13
Acurácia: 0.6971040189125296
Melhor K: 9
Melhor Acurácia: 0.6971040189125296

Acurácia final: 0.7777777777777778
